# So sánh chất lượng giữa các lần thử nghiệm

Notebook này **đọc** báo cáo đã ghi sẵn trên đĩa, không huấn luyện lại. Chạy trước (ít nhất một lần,
không cần đủ cả mười biến thể):

```bash
uv run python -m app.scripts.train_risk_model_experiments
```

Nguồn dữ liệu: `evaluation_metrics.csv` và `evaluation_report.json` của mô hình sản xuất
(`RISK_MODEL_DIR`) và của mọi thư mục con trong `experiments/` mà `train_risk_model_experiments.py`
đã ghi. Ghép lại thành bảng và biểu đồ so sánh Accuracy/F1/ROC-AUC theo (biến thể, thuật toán, mốc dự
đoán).

**Lưu notebook với ô kết quả đã xoá sạch** — cùng quy ước với `risk_model_analysis.ipynb`: số liệu đã
nằm ở JSON/CSV, giữ thêm bản trong notebook chỉ làm git diff nhiễu.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from app.core.config import settings
from app.risk.predictor import CHECKPOINTS
from app.risk.training import METRICS_FILENAME, REPORT_FILENAME

BASELINE_DIR = Path(settings.RISK_MODEL_DIR)
EXPERIMENTS_DIR = BASELINE_DIR.parent / "experiments"  # sibling của RISK_MODEL_DIR, khớp #45

variant_dirs = sorted(
    path for path in EXPERIMENTS_DIR.glob("*") if (path / METRICS_FILENAME).exists()
)

print(f"Baseline (sản xuất): {BASELINE_DIR}")
print(f"Tìm thấy {len(variant_dirs)} biến thể đã có báo cáo trong {EXPERIMENTS_DIR}:")
for path in variant_dirs:
    print(f"  - {path.name}")

## 1. Bảng so sánh Accuracy / F1 / ROC-AUC

`roc_auc` để trống ở biến thể/thuật toán/mốc nào tập kiểm tra chỉ còn một lớp nhãn (#41) — không
phải lỗi đọc tệp. `accuracy` và `f1` đo tại ngưỡng đề xuất riêng của từng biến thể (mỗi biến thể có
tập kiểm định, và do đó ngưỡng, khác nhau); `roc_auc` không phụ thuộc ngưỡng nào — ba cột không cùng
"thước đo" nên đừng cộng dồn hay lấy trung bình chúng với nhau.

In [ ]:
def read_metrics(directory: Path, label: str) -> pd.DataFrame:
    frame = pd.read_csv(directory / METRICS_FILENAME)
    frame.insert(0, "biến thể", label)
    return frame[["biến thể", "algorithm", "checkpoint", "accuracy", "f1", "roc_auc"]]


frames = [read_metrics(BASELINE_DIR, "baseline (sản xuất)")]
frames += [read_metrics(path, path.name) for path in variant_dirs]

metrics = pd.concat(frames, ignore_index=True).rename(
    columns={"algorithm": "thuật toán", "checkpoint": "mốc"}
)

metrics.pivot_table(
    index=["biến thể", "thuật toán"], columns="mốc", values=["accuracy", "f1", "roc_auc"]
)

## 2. Ngữ cảnh mỗi biến thể

Thuật toán nào được chọn, ngưỡng đề xuất bao nhiêu, và biến thể đó có đạt mục tiêu F1 >= 0,30 ở mốc
đặt hàng hay không — cùng tiêu chí chọn thuật toán với lệnh huấn luyện sản xuất (ADR-0008), áp dụng
riêng cho từng biến thể.

In [ ]:
def read_context(directory: Path, label: str) -> dict:
    report = json.loads((directory / REPORT_FILENAME).read_text("utf-8"))
    return {
        "biến thể": label,
        "thuật toán được chọn": report["selected_algorithm"],
        "ngưỡng đề xuất": report["suggested_risk_threshold"],
        "F1 ở mốc đặt hàng": report["f1_at_order_placed"],
        "đạt mục tiêu F1": report["meets_f1_target"],
    }


context = pd.DataFrame(
    [read_context(BASELINE_DIR, "baseline (sản xuất)")]
    + [read_context(path, path.name) for path in variant_dirs]
)
context

## 3. Biểu đồ so sánh theo mốc dự đoán

Mỗi thanh là một (biến thể, thuật toán), sắp theo giá trị chỉ số, một biểu đồ con cho mỗi mốc dự
đoán. Biến thể nào chỉ thử một thuật toán (ví dụ các biến thể LightGBM) chỉ có một thanh.

In [ ]:
metrics["nhãn"] = metrics["biến thể"] + " / " + metrics["thuật toán"]

for column in ("accuracy", "f1", "roc_auc"):
    height = 0.35 * metrics["nhãn"].nunique() + 1
    fig, axes = plt.subplots(1, len(CHECKPOINTS), figsize=(6 * len(CHECKPOINTS), height), sharey=True)

    for axis, checkpoint in zip(axes, CHECKPOINTS):
        part = metrics[metrics["mốc"] == checkpoint].dropna(subset=[column]).sort_values(column)
        axis.barh(part["nhãn"], part[column])
        axis.set_title(checkpoint)
        axis.set_xlabel(column)

    fig.suptitle(f"So sánh {column} theo biến thể / thuật toán")
    plt.tight_layout()
    plt.show()

## 4. Khuyến nghị

**Không có biến thể nào trong 11 biến thể đã thử đáng để đổi cấu hình đang phục vụ đơn thật.** Giữ
nguyên `sklearn_quantile` theo ADR-0008.

> `feature_freight_ratio` là biến thể thứ 11, hướng "thêm đặc trưng" của #47 — không nằm trong
> `VARIANTS` của `train_risk_model_experiments.py` (hướng này không tham số hoá được, xem #47), nên
> chạy lại lệnh ở ô đầu notebook sẽ **không** tái tạo được thư mục này. Đây là một lần chạy tay riêng
> (sửa tạm `features.py` thêm cột `freight_ratio`, gọi `train()`, rồi hoàn tác mã nguồn) — kết quả
> vẫn còn nguyên trên đĩa ở `experiments/feature_freight_ratio/` dù mã nguồn sinh ra nó đã hoàn tác.

So các biến thể với `baseline` ở mốc **đặt hàng** (mốc khó nhất, ADR-0008 dùng để chọn thuật toán),
F1 tại ngưỡng đề xuất riêng của từng biến thể — `baseline` = 0,1972:

| Hướng | Biến thể tốt nhất | F1 (đặt hàng) | So với baseline |
| --- | --- | --- | --- |
| Siêu tham số | `hyperparam_shallower_trees` | 0,1994 | +0,0022 — trong biên độ nhiễu, không đáng kể |
| LightGBM | `lightgbm_deeper` | 0,1902 | −0,0070 — kém hơn baseline |
| Tỷ lệ chia tập | `split_80_10_10` | 0,2332 | +0,0360 (nhìn có vẻ ăn nhất) — **nhưng xem lý do dưới đây** |
| Đặc trưng mới | `feature_freight_ratio` | 0,1849 | −0,0123 — kém hơn baseline |

### Vì sao không chọn `split_80_10_10` dù số đẹp nhất

Con số đó không so sánh công bằng: tập kiểm tra của `split_80_10_10` chỉ còn 9.628 dòng (so với
14.441 của baseline), lệch hẳn sang giai đoạn cuối (18/07–29/08/2018), và có `late_rate` 5,29% thay
vì 4,29% của baseline — một tập kiểm tra khác hẳn, không phải cùng một "đề thi".

Có một phép so sánh sòng phẳng chứng minh điều này: `split_75_10_15` (75/10/15) tình cờ cho **đúng
cùng tập kiểm tra** với baseline (70/15/15) — cả hai cùng cắt ở mốc 85% dữ liệu (70+15 = 75+10 = 85),
nên cùng 14.441 dòng, cùng khoảng thời gian, cùng `late_rate` 4,29%. Vậy mà F1 của nó chỉ 0,1503,
**thấp hơn hẳn** baseline — cùng một đề thi, chỉ đổi tỷ lệ train/validation, kết quả tệ đi rõ rệt
(nhiều khả năng vì tập validation nhỏ/lệch hơn chọn ra một ngưỡng kém). Đây là bằng chứng trực tiếp
rằng đổi tỷ lệ chia tập không đáng tin cậy để đánh giá "mô hình tốt hơn" — chênh lệch ở hướng này dễ
là do đổi luôn độ khó của đề thi (kích thước và thành phần tập kiểm tra), không phải do mô hình học
tốt hơn.

### Từng hướng, ngắn gọn

- **Siêu tham số:** biến thể tốt nhất chỉ nhỉnh hơn 0,0022 — không đáng kể so với biên độ dao động
  giữa các biến thể khác (từ 0,1713 đến 0,1994). `hyperparam_slow_learning_rate` còn rõ ràng kém
  hơn (0,1713). Đánh đổi nếu đổi: rẻ nhất trong bốn hướng (chỉ đổi vài số trong `candidates.py`,
  không dependency mới), nhưng không có gì để đổi lấy.
- **LightGBM:** cả hai biến thể đều thấp hơn baseline ở F1 lẫn ROC-AUC (mốc đặt hàng: 0,7678–0,7801
  so với 0,7877 của baseline). Dependency đã trả giá từ #44 (đã trong `pyproject.toml`/`uv.lock`),
  nhưng dùng chính thức nghĩa là nuôi hai thư viện gradient boosting song song
  (LightGBM + HistGradientBoosting của `sklearn_quantile`) mà không có lợi ích đo được để biện minh.
- **Tỷ lệ chia tập:** xem phân tích trên — số đẹp nhất trong bốn hướng nhưng là kết quả giả do đổi
  thành phần tập kiểm tra, không phải mô hình tốt hơn. Đánh đổi nếu đổi: rẻ nhất về code, nhưng đổi
  luôn ý nghĩa "tập kiểm tra là gì" — khó biện minh cho quản lý hậu cần bằng một con số suông.
- **Đặc trưng `freight_ratio`:** F1 giảm nhẹ ở ngưỡng đã chọn (0,1849 so với 0,1972), dù ROC-AUC ở
  cả ba mốc nhích lên chút ít cho `sklearn_quantile` (ví dụ mốc đặt hàng: 0,7909 so với 0,7877) —
  tín hiệu lẫn lộn, khớp với dự đoán rằng tỷ lệ phí ship/giá trị phần lớn trùng lặp thông tin đã có
  sẵn ở `total_price`/`total_freight`/`distance_km`. Đánh đổi nếu đổi: **đắt nhất** trong bốn hướng
  để đưa vào chính thức — không chỉ đổi cấu hình, còn phải sửa `predictor.py` (đường dự đoán thật)
  để tính đặc trưng này lúc phục vụ, thêm bề mặt bảo trì mà ba hướng kia không có.

### Ghi chú thêm

Không biến thể nào (kể cả baseline) đạt mục tiêu F1 >= 0,30 ở mốc đặt hàng — nằm ngoài phạm vi
ticket này (ticket không đặt đích cứng phải đạt mức nào). Nếu muốn tiếp tục thử hướng tỷ lệ chia tập
ở một ticket khác, nên giữ cố định kích thước và khoảng thời gian của tập kiểm tra khi so sánh, thay
vì chỉ đổi điểm cắt — cách làm hiện tại (đổi điểm cắt) khiến "tập kiểm tra" đổi luôn theo từng biến
thể và không so sánh được công bằng.
